# Skin Lesion Classification — DenseNet121 + Attention Block + Clinical MLP (TensorFlow)

**Everything is in this notebook** — no other project files are needed. Run the cells top to bottom.

```
Image ─► Preprocessing ─► DenseNet121 (ImageNet) ─► Attention Block ─► GAP ──┐
                             7×7×1024                 7×7×1024          1024  ├─► Concat ─► Dense(256) ─► FC(7, softmax)
Clinical data (age, sex, localization) ─► MLP (128 → 64) ─────────────────── 64 ──┘
```

**Attention Block** (input F = H×W×C):

| Path | Operation | Output |
|---|---|---|
| 1 | Conv 1×1 (C → 1) | H×W×1 |
| 2 | Max-pool across channels | H×W×1 |
| 3 | Avg-pool across channels = A (H×W×1) → reshape 1×HW → sigmoid → reshape back = gate → **A ⊗ gate** | H×W×1 |
| fuse | Concat → H×W×3 → Conv 1×1 → sigmoid = attention map M | H×W×1 |
| out | F ⊗ M (broadcast over channels) | H×W×C |

**GPU note (Windows):** TensorFlow ≥ 2.11 has no GPU support on native Windows. Use a **Python 3.10** environment with **TensorFlow 2.10** (setup in section 0), or WSL2. The code works on TF 2.10 (Keras 2) and TF 2.16+ (Keras 3).

## 0. One-time environment setup (run in a terminal / Anaconda Prompt, not here)

```bash
conda create -n skin python=3.10 -y
conda activate skin
conda install -c conda-forge cudatoolkit=11.2 cudnn=8.1.0 -y
pip install "tensorflow==2.10.1" "numpy<2" pandas scikit-learn matplotlib notebook ipykernel
python -m ipykernel install --user --name skin --display-name "Python (skin)"
```
Then open this notebook and choose the kernel **Python (skin)**.

## 1. Imports and GPU check

In [ ]:
import os, json, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support, roc_auc_score)

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
GPUS = tf.config.list_physical_devices("GPU")
for g in GPUS:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError:
        pass
print("GPU:", GPUS if GPUS else "NONE -> running on CPU (slow). See section 0.")

# Keras 3 saves weights as .weights.h5; Keras 2 (TF 2.10) uses the TF-checkpoint
# format, because its .h5 weight order changes when layers are frozen/unfrozen.
KERAS3 = int(keras.__version__.split(".")[0]) >= 3

## 2. Configuration — **edit `DATA_ROOT`**

In [ ]:
# Folder containing HAM10000_metadata.csv, HAM10000_images_part_1/, HAM10000_images_part_2/
DATA_ROOT = r"C:\Users\Dilip Prajapati\Desktop\MInor Project_Skin\Dataset\Data"   # <-- EDIT

METADATA_CSV = os.path.join(DATA_ROOT, "HAM10000_metadata.csv")
IMAGE_DIRS = [os.path.join(DATA_ROOT, "HAM10000_images_part_1"),
              os.path.join(DATA_ROOT, "HAM10000_images_part_2")]
CLASS_NAMES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = 224                 # DenseNet121 native resolution, fits 6 GB VRAM

# Split mode:
#   "strict" -> 70/15/15 split grouped by lesion_id (no lesion in two splits), dx_type excluded
#   "paper"  -> 70/30 image-level split, no validation set, dx_type included (inflated numbers)
SPLIT_MODE = "strict"
USE_DX_TYPE = SPLIT_MODE == "paper"
TRAIN_FRAC = 0.70
VAL_FRAC = 0.0 if SPLIT_MODE == "paper" else 0.15
TEST_FRAC = 0.30 if SPLIT_MODE == "paper" else 0.15
SEED = 42

# Class imbalance: "oversample" (equal images per class each epoch) | "class_weight" | "none"
BALANCE_STRATEGY = "oversample"
TARGET_PER_CLASS = 2000        # 7 x 2000 = 14,000 training images per epoch

# Model
CLINICAL_MLP_UNITS = (128, 64)
FUSION_UNITS = 256
DROPOUT = 0.4

# Training (tuned for RTX 3050 6 GB)
BATCH_SIZE = 16                # lower to 8 if you get out-of-memory
WARMUP_EPOCHS = 5              # phase 1: backbone frozen
WARMUP_LR = 1e-3
FINE_TUNE_EPOCHS = 30          # phase 2: fine-tune backbone
FINE_TUNE_LR = 1e-4
FINE_TUNE_LAYERS = 150         # last N DenseNet layers unfrozen (None = all)
EARLY_STOP_PATIENCE = 8
LABEL_SMOOTHING = 0.05
USE_MIXED_PRECISION = True     # float16 compute on GPU
CACHE_IMAGES_IN_RAM = False    # True = faster epochs after the first (~1.5 GB RAM)

OUTPUT_DIR = "outputs_" + SPLIT_MODE

tf.keras.utils.set_random_seed(SEED)
if GPUS and USE_MIXED_PRECISION:
    keras.mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision ON")

assert os.path.exists(METADATA_CSV), f"Not found: {METADATA_CSV} -- fix DATA_ROOT"
for d in IMAGE_DIRS:
    print(d, "->", len(os.listdir(d)) if os.path.isdir(d) else "MISSING", "files")

## 3. Load metadata and split
`strict` mode splits by **lesion_id** so the same lesion never appears in both train and test.

In [ ]:
def load_and_split_metadata():
    df = pd.read_csv(METADATA_CSV)
    if SPLIT_MODE == "paper":
        tr, te = train_test_split(df, train_size=TRAIN_FRAC, stratify=df["dx"], random_state=SEED)
        return tr.reset_index(drop=True), None, te.reset_index(drop=True)
    lesions = df.drop_duplicates(subset="lesion_id")[["lesion_id", "dx"]]
    tr_l, tmp_l = train_test_split(lesions, train_size=TRAIN_FRAC, stratify=lesions["dx"],
                                   random_state=SEED)
    va_l, te_l = train_test_split(tmp_l, train_size=VAL_FRAC / (VAL_FRAC + TEST_FRAC),
                                  stratify=tmp_l["dx"], random_state=SEED)
    pick = lambda ids: df[df.lesion_id.isin(ids.lesion_id)].reset_index(drop=True)
    return pick(tr_l), pick(va_l), pick(te_l)


train_df, val_df, test_df = load_and_split_metadata()
print(f"train {len(train_df)} | val {0 if val_df is None else len(val_df)} | test {len(test_df)}")
pd.DataFrame({s: d["dx"].value_counts() for s, d in
              [("train", train_df), ("val", val_df), ("test", test_df)] if d is not None})

## 4. Clinical features (the "Clinical data" input)
age → median-imputed + standardised; sex, localization → one-hot ("unknown" = missing → mode).
All statistics are fitted on **train only**.

In [ ]:
CAT_COLS = ["sex", "localization"] + (["dx_type"] if USE_DX_TYPE else [])


def fit_clinical_encoder(df):
    enc = {"age_median": float(df["age"].median()), "categories": {}, "modes": {}}
    age = df["age"].fillna(enc["age_median"])
    enc["age_mean"], enc["age_std"] = float(age.mean()), float(age.std() or 1.0)
    for col in CAT_COLS:
        vals = df[col].replace("unknown", np.nan)
        enc["modes"][col] = str(vals.mode()[0])
        enc["categories"][col] = sorted(vals.fillna(enc["modes"][col]).astype(str).unique())
    enc["feature_names"] = ["age"] + [f"{c}_{v}" for c in CAT_COLS for v in enc["categories"][c]]
    return enc


def transform_clinical(df, enc):
    age = df["age"].fillna(enc["age_median"]).to_numpy(dtype=np.float32)
    parts = [((age - enc["age_mean"]) / enc["age_std"])[:, None]]
    for col, cats in enc["categories"].items():
        vals = df[col].replace("unknown", np.nan).fillna(enc["modes"][col]).astype(str)
        idx = {c: i for i, c in enumerate(cats)}
        onehot = np.zeros((len(df), len(cats)), dtype=np.float32)
        for r, v in enumerate(vals):
            if v in idx:
                onehot[r, idx[v]] = 1.0
        parts.append(onehot)
    return np.concatenate(parts, axis=1).astype(np.float32)


def labels_of(df):
    m = {c: i for i, c in enumerate(CLASS_NAMES)}
    return df["dx"].map(m).to_numpy(dtype=np.int32)


encoder = fit_clinical_encoder(train_df)
X_clin = {"train": transform_clinical(train_df, encoder), "test": transform_clinical(test_df, encoder)}
Y = {"train": labels_of(train_df), "test": labels_of(test_df)}
if val_df is not None:
    X_clin["val"], Y["val"] = transform_clinical(val_df, encoder), labels_of(val_df)
CLINICAL_DIM = X_clin["train"].shape[1]
print(CLINICAL_DIM, "clinical features:", encoder["feature_names"])

## 5. Image pipeline (tf.data) + augmentation + class balancing
In `oversample` mode each class gets its own shuffled, repeating stream, sampled with equal
probability — so every epoch sees ~`TARGET_PER_CLASS` images per class, with a fresh random
augmentation on every repeat.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

lookup = {}
for d in IMAGE_DIRS:
    for f in (os.listdir(d) if os.path.isdir(d) else []):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            lookup[os.path.splitext(f)[0]] = os.path.join(d, f)


def image_paths_for(df):
    missing = [i for i in df["image_id"] if i not in lookup]
    if missing:
        raise FileNotFoundError(f"{len(missing)} images missing, e.g. {missing[:3]}")
    return np.array([lookup[i] for i in df["image_id"]])


PATHS = {"train": image_paths_for(train_df), "test": image_paths_for(test_df)}
if val_df is not None:
    PATHS["val"] = image_paths_for(val_df)


def load_image(path, size=IMG_SIZE):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (size, size), antialias=True)
    return tf.cast(img, tf.float32)                 # 0..255; normalised inside the model


def augment(img):
    size = tf.shape(img)[0]
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    crop = tf.cast(tf.cast(size, tf.float32) * tf.random.uniform([], 0.80, 1.0), tf.int32)
    img = tf.image.random_crop(img, tf.stack([crop, crop, 3]))     # random zoom
    img = tf.image.resize(img, tf.stack([size, size]))
    img = img / 255.0
    img = tf.image.random_brightness(img, 0.10)
    img = tf.image.random_contrast(img, 0.9, 1.1)
    img = tf.image.random_saturation(img, 0.9, 1.1)
    img = tf.image.random_hue(img, 0.02)
    return tf.clip_by_value(img, 0.0, 1.0) * 255.0


def _base_ds(paths, clin, labels, load_images=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, clin, tf.one_hot(labels, NUM_CLASSES)))
    def _load(p, c, y):
        img = load_image(p) if load_images else tf.zeros((IMG_SIZE, IMG_SIZE, 3))
        return img, c, y
    return ds.map(_load, num_parallel_calls=AUTOTUNE)


def _to_inputs(img, c, y):
    return {"image": img, "clinical": c}, y


def make_eval_ds(split, batch_size=BATCH_SIZE, load_images=True):
    ds = _base_ds(PATHS[split], X_clin[split], Y[split], load_images)
    if CACHE_IMAGES_IN_RAM:
        ds = ds.cache()
    return ds.map(_to_inputs).batch(batch_size).prefetch(AUTOTUNE)


def make_train_ds(batch_size=BATCH_SIZE, load_images=True):
    paths, clin, labels = PATHS["train"], X_clin["train"], Y["train"]
    if BALANCE_STRATEGY == "oversample":
        streams = []
        for k in range(NUM_CLASSES):
            m = labels == k
            if not m.any():
                continue
            s = _base_ds(paths[m], clin[m], labels[m], load_images)
            if CACHE_IMAGES_IN_RAM:
                s = s.cache()
            streams.append(s.shuffle(int(m.sum()), seed=SEED + k).repeat())
        ds = tf.data.Dataset.sample_from_datasets(streams, [1.0 / len(streams)] * len(streams), seed=SEED)
        steps = int(np.ceil(TARGET_PER_CLASS * len(streams) / batch_size))
    else:
        ds = _base_ds(paths, clin, labels, load_images)
        if CACHE_IMAGES_IN_RAM:
            ds = ds.cache()
        ds = ds.shuffle(len(labels), seed=SEED).repeat()
        steps = int(np.ceil(len(labels) / batch_size))
    aug = (lambda i, c, y: (augment(i), c, y)) if load_images else (lambda i, c, y: (i, c, y))
    ds = (ds.map(aug, num_parallel_calls=AUTOTUNE).map(_to_inputs)
            .batch(batch_size, drop_remainder=True).prefetch(AUTOTUNE))
    return ds, steps


# preview a few augmented training images
_prev, _ = make_train_ds(batch_size=8)
_x, _y = next(iter(_prev))
fig, axes = plt.subplots(1, 8, figsize=(18, 3))
for i, ax in enumerate(axes):
    ax.imshow(_x["image"][i].numpy().astype("uint8"))
    ax.set_title(CLASS_NAMES[int(np.argmax(_y[i]))]); ax.axis("off")
plt.show()

## 6. Attention Block (exactly as drawn)

In [ ]:
class AttentionBlock(keras.layers.Layer):
    """Input F: H x W x C.  Returns [F * M (H x W x C), attention map M (H x W x 1)]."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.branch_conv = keras.layers.Conv2D(1, 1, padding="same", name="branch_conv1x1")
        self.fuse_conv = keras.layers.Conv2D(1, 1, padding="same", name="fuse_conv1x1")

    def call(self, x):
        # Path 1: Conv 1x1 -> H x W x 1
        b1 = self.branch_conv(x)
        # Path 2: Max pool across channels -> H x W x 1
        b2 = tf.reduce_max(x, axis=-1, keepdims=True)
        # Path 3: Avg pool across channels -> A (H x W x 1)
        a = tf.reduce_mean(x, axis=-1, keepdims=True)
        s = tf.shape(a)
        gate = tf.sigmoid(tf.reshape(a, [s[0], 1, s[1] * s[2]]))   # reshape 1 x HW, sigmoid
        gate = tf.reshape(gate, s)                                  # back to H x W x 1
        b3 = a * gate                                               # A (x) gate -> H x W x 1
        # Concat -> H x W x 3 -> Conv 1x1 -> H x W x 1 -> sigmoid
        m = tf.sigmoid(self.fuse_conv(tf.concat([b1, b2, b3], axis=-1)))
        # Multiply with the original input -> H x W x C
        return [x * m, m]

    def compute_output_shape(self, input_shape):
        return [tuple(input_shape), tuple(input_shape[:-1]) + (1,)]


# shape check
_o, _m = AttentionBlock()(tf.random.normal((2, 7, 7, 1024)))
print("in (2,7,7,1024) -> out", _o.shape, "| map", _m.shape)

## 7. Backbone, clinical MLP and the full model

In [ ]:
class Preprocess(keras.layers.Layer):
    """[0,255] RGB -> ImageNet normalisation the backbone was trained with."""

    def __init__(self, backbone="densenet121", **kwargs):
        super().__init__(**kwargs)
        self.backbone = backbone

    def call(self, x):
        x = tf.cast(x, self.compute_dtype)
        if self.backbone == "densenet121":
            mean = tf.constant([0.485, 0.456, 0.406], dtype=x.dtype)
            std = tf.constant([0.229, 0.224, 0.225], dtype=x.dtype)
            return (x / 255.0 - mean) / std
        return x / 127.5 - 1.0          # mobilenetv2


def build_backbone(name, weights="imagenet"):
    shape = (IMG_SIZE, IMG_SIZE, 3)
    if name == "densenet121":
        return keras.applications.DenseNet121(include_top=False, weights=weights, input_shape=shape)
    if name == "mobilenetv2":
        return keras.applications.MobileNetV2(include_top=False, weights=weights, input_shape=shape)
    raise ValueError(name)


def clinical_mlp(x):
    for i, u in enumerate(CLINICAL_MLP_UNITS):
        x = keras.layers.Dense(u, name=f"clinical_dense{i+1}")(x)
        x = keras.layers.BatchNormalization(name=f"clinical_bn{i+1}")(x)
        x = keras.layers.Activation("relu", name=f"clinical_relu{i+1}")(x)
        if i < len(CLINICAL_MLP_UNITS) - 1:
            x = keras.layers.Dropout(0.3, name=f"clinical_drop{i+1}")(x)
    return x


def image_branch(image_in, backbone_name, use_attention, weights):
    x = Preprocess(backbone_name, name="preprocess")(image_in)
    x = build_backbone(backbone_name, weights)(x, training=False)   # BN kept in inference mode
    if use_attention:
        x, _ = AttentionBlock(name="attention_block")(x)
    x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
    return keras.layers.Dropout(DROPOUT, name="image_dropout")(x)


def head(x):
    x = keras.layers.Dense(FUSION_UNITS, name="fusion_dense")(x)
    x = keras.layers.BatchNormalization(name="fusion_bn")(x)
    x = keras.layers.Activation("relu", name="fusion_relu")(x)
    x = keras.layers.Dropout(DROPOUT, name="fusion_dropout")(x)
    x = keras.layers.Dense(NUM_CLASSES, name="fc_logits")(x)
    return keras.layers.Activation("softmax", dtype="float32", name="predictions")(x)


def build_model(model_type="fusion", backbone="densenet121", use_attention=True, weights="imagenet"):
    """model_type: 'fusion' (full design) | 'image_only' | 'clinical_only'"""
    image_in = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    clinical_in = keras.Input((CLINICAL_DIM,), name="clinical")
    tag = "attn" if use_attention else "noattn"
    if model_type == "fusion":
        fused = keras.layers.Concatenate(name="concat")(
            [image_branch(image_in, backbone, use_attention, weights), clinical_mlp(clinical_in)])
        out, name = head(fused), f"{backbone}_{tag}_fusion"
    elif model_type == "image_only":
        out = head(image_branch(image_in, backbone, use_attention, weights))
        name = f"{backbone}_{tag}_image_only"
    elif model_type == "clinical_only":
        out, name = head(clinical_mlp(clinical_in)), "clinical_only"
    else:
        raise ValueError(model_type)
    return keras.Model({"image": image_in, "clinical": clinical_in}, out, name=name)


def get_backbone(model):
    return next((l for l in model.layers if isinstance(l, keras.Model)), None)


def set_backbone_trainable(backbone, trainable, last_n=None):
    backbone.trainable = trainable
    if not trainable:
        return
    cutoff = 0 if last_n is None else max(0, len(backbone.layers) - last_n)
    for i, layer in enumerate(backbone.layers):
        layer.trainable = (not isinstance(layer, keras.layers.BatchNormalization)) and i >= cutoff


build_model(weights=None).summary(line_length=110)

## 8. Evaluation helpers

In [ ]:
def weights_file(run_path, which):
    return os.path.join(run_path, f"{which}.weights.h5" if KERAS3 else f"{which}.ckpt")


def weights_exist(p):
    return os.path.exists(p) or os.path.exists(p + ".index")


def load_w(model, path):
    st = model.load_weights(path)
    if hasattr(st, "expect_partial"):
        st.expect_partial()


def evaluate_and_report(y_true, probs, run_path, tag="test", image_ids=None):
    y_pred = probs.argmax(1)
    labels = list(range(NUM_CLASSES))
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(y_true, probs, multi_class="ovr", average="macro", labels=labels)
    except ValueError:
        auc = float("nan")
    metrics = {k: float(v) for k, v in {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": p, "recall_macro": r, "f1_macro": f1, "roc_auc_macro_ovr": auc}.items()}
    json.dump(metrics, open(os.path.join(run_path, f"{tag}_metrics.json"), "w"), indent=2)
    report = classification_report(y_true, y_pred, labels=labels, target_names=CLASS_NAMES,
                                   digits=4, zero_division=0)
    open(os.path.join(run_path, f"{tag}_classification_report.txt"), "w").write(report)
    print(report)
    print(" | ".join(f"{k}: {v:.4f}" for k, v in metrics.items()))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(labels); ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticks(labels); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Confusion matrix ({tag})")
    for i in labels:
        for j in labels:
            ax.text(j, i, f"{cm[i, j]}\n{cmn[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if cmn[i, j] > 0.5 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046); fig.tight_layout()
    fig.savefig(os.path.join(run_path, f"{tag}_confusion_matrix.png"), dpi=150); plt.show()

    if image_ids is not None:
        out = pd.DataFrame(probs, columns=[f"p_{c}" for c in CLASS_NAMES])
        out.insert(0, "image_id", image_ids)
        out.insert(1, "true", [CLASS_NAMES[i] for i in y_true])
        out.insert(2, "pred", [CLASS_NAMES[i] for i in y_pred])
        out.to_csv(os.path.join(run_path, f"{tag}_predictions.csv"), index=False)
    return metrics


def plot_history(history_csv, run_path):
    h = pd.read_csv(history_csv)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, key in zip(axes, ["loss", "accuracy"]):
        ax.plot(h["epoch"] + 1, h[key], label="train")
        if f"val_{key}" in h:
            ax.plot(h["epoch"] + 1, h[f"val_{key}"], label="val")
        ax.set_title(key); ax.set_xlabel("epoch"); ax.grid(alpha=0.3); ax.legend()
    fig.tight_layout(); fig.savefig(os.path.join(run_path, "training_curves.png"), dpi=150); plt.show()

## 9. Training function
Phase 1: backbone frozen (train attention block + MLP + head). Phase 2: unfreeze the last
`FINE_TUNE_LAYERS` DenseNet layers with a lower learning rate. Weights are saved every epoch;
if the kernel dies, call the same `run_experiment(...)` with `resume=True`.

In [ ]:
def run_experiment(model_type="fusion", backbone="densenet121", use_attention=True,
                   warmup_epochs=WARMUP_EPOCHS, finetune_epochs=FINE_TUNE_EPOCHS,
                   batch_size=BATCH_SIZE, run_name=None, resume=False, pretrained=True):
    keras.backend.clear_session()
    load_images = model_type != "clinical_only"
    train_ds, steps = make_train_ds(batch_size, load_images)
    val_ds = make_eval_ds("val", batch_size, load_images) if val_df is not None else None
    test_ds = make_eval_ds("test", batch_size, load_images)

    class_weight = None
    if BALANCE_STRATEGY == "class_weight":
        present = np.unique(Y["train"])
        class_weight = dict(zip(present.tolist(),
                                compute_class_weight("balanced", classes=present, y=Y["train"]).tolist()))

    model = build_model(model_type, backbone, use_attention, "imagenet" if pretrained else None)
    run_name = run_name or model.name
    out = os.path.join(OUTPUT_DIR, run_name)
    os.makedirs(out, exist_ok=True)
    json.dump(encoder, open(os.path.join(out, "clinical_encoder.json"), "w"), indent=2)
    json.dump({"model_type": model_type, "backbone": backbone, "use_attention": use_attention,
               "clinical_dim": CLINICAL_DIM}, open(os.path.join(out, "run_config.json"), "w"), indent=2)
    print(f"Run: {run_name} -> {out}")

    has_val = val_ds is not None
    monitor = "val_loss" if has_val else "loss"
    latest_w, best_w = weights_file(out, "latest"), weights_file(out, "best")
    state_path, history_csv = os.path.join(out, "state.json"), os.path.join(out, "history.csv")

    state = {"epochs_done": 0, "best": None}
    if resume and os.path.exists(state_path) and weights_exist(latest_w):
        state = json.load(open(state_path))
        load_w(model, latest_w)
        print(f"Resuming after epoch {state['epochs_done']}")
    elif os.path.exists(history_csv):
        os.remove(history_csv)

    class SaveState(keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            model.save_weights(latest_w)
            cur = (logs or {}).get(monitor)
            if cur is not None and (state["best"] is None or cur < state["best"]):
                state["best"] = float(cur)
            state["epochs_done"] = epoch + 1
            json.dump(state, open(state_path, "w"))

    def callbacks(early_stop=True):
        cbs = [SaveState(), keras.callbacks.CSVLogger(history_csv, append=True)]
        if has_val:
            kw = dict(monitor="val_loss", save_best_only=True, save_weights_only=True, verbose=1)
            if state["best"] is not None:
                kw["initial_value_threshold"] = state["best"]
            cbs += [keras.callbacks.ModelCheckpoint(best_w, **kw),
                    keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.3, patience=3,
                                                      min_lr=1e-7, verbose=1)]
            if early_stop:
                cbs.append(keras.callbacks.EarlyStopping("val_loss", patience=EARLY_STOP_PATIENCE,
                                                         verbose=1))
        return cbs

    def compile_(lr):
        model.compile(optimizer=keras.optimizers.Adam(lr),
                      loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
                      metrics=["accuracy", keras.metrics.AUC(name="auc")])

    fit_kw = dict(steps_per_epoch=steps, validation_data=val_ds, class_weight=class_weight, verbose=1)
    t0 = time.time()
    if model_type == "clinical_only":
        compile_(WARMUP_LR)
        model.fit(train_ds, epochs=warmup_epochs + finetune_epochs,
                  initial_epoch=state["epochs_done"], callbacks=callbacks(), **fit_kw)
    else:
        bb = get_backbone(model)
        if state["epochs_done"] < warmup_epochs:
            set_backbone_trainable(bb, False)
            compile_(WARMUP_LR)
            print("\n=== Phase 1: backbone frozen")
            model.fit(train_ds, epochs=warmup_epochs, initial_epoch=state["epochs_done"],
                      callbacks=callbacks(early_stop=False), **fit_kw)
        set_backbone_trainable(bb, True, FINE_TUNE_LAYERS)
        compile_(FINE_TUNE_LR)
        print(f"\n=== Phase 2: fine-tuning last {FINE_TUNE_LAYERS or 'ALL'} backbone layers")
        model.fit(train_ds, epochs=warmup_epochs + finetune_epochs,
                  initial_epoch=max(warmup_epochs, state["epochs_done"]), callbacks=callbacks(), **fit_kw)

    model.save_weights(weights_file(out, "final"))
    print(f"Training time: {(time.time() - t0) / 60:.1f} min")
    plot_history(history_csv, out)

    if has_val and weights_exist(best_w):
        load_w(model, best_w); print("Test set -> BEST-validation weights")
    else:
        print("Test set -> FINAL weights")
    probs = model.predict(test_ds, verbose=1)
    evaluate_and_report(Y["test"], probs, out, "test", test_df["image_id"].tolist())
    return model, out

## 10. Quick pipeline test (1 + 1 epochs, few steps) — run this first

In [ ]:
_saved = TARGET_PER_CLASS
TARGET_PER_CLASS = 32       # tiny epoch just to test the pipeline
_m, _ = run_experiment(warmup_epochs=1, finetune_epochs=1, run_name="smoke_test")
TARGET_PER_CLASS = _saved

## 11. Train the full model (main result)

In [ ]:
model, RUN = run_experiment(model_type="fusion", backbone="densenet121", use_attention=True)
# interrupted?  ->  model, RUN = run_experiment(resume=True)

## 12. Attention maps on test images

In [ ]:
def show_attention(model, n_per_class=2):
    viz = keras.Model(model.inputs, [model.get_layer("attention_block").output[1], model.output])
    sample = pd.concat([g.sample(min(len(g), n_per_class), random_state=1)
                        for _, g in test_df.groupby("dx")]).reset_index(drop=True)
    imgs = np.stack([load_image(lookup[i]).numpy() for i in sample["image_id"]])
    maps, probs = viz.predict({"image": imgs, "clinical": transform_clinical(sample, encoder)}, verbose=0)
    y = labels_of(sample)
    n = len(sample)
    fig, axes = plt.subplots(n // 2 + n % 2, 4, figsize=(12, 3 * (n // 2 + n % 2)))
    axes = np.array(axes).reshape(-1, 4)
    for ax in axes.ravel():
        ax.axis("off")
    for i in range(n):
        r, c = divmod(i, 2)
        m = tf.image.resize(maps[i].astype("float32"), (IMG_SIZE, IMG_SIZE)).numpy()[..., 0]
        m = (m - m.min()) / (m.max() - m.min() + 1e-8)
        pred = int(probs[i].argmax())
        axes[r, 2 * c].imshow(imgs[i].astype("uint8"))
        axes[r, 2 * c].set_title(f"true: {CLASS_NAMES[y[i]]}", fontsize=9)
        axes[r, 2 * c + 1].imshow(imgs[i].astype("uint8"))
        axes[r, 2 * c + 1].imshow(m, cmap="jet", alpha=0.45)
        axes[r, 2 * c + 1].set_title(f"pred: {CLASS_NAMES[pred]} ({probs[i, pred]:.2f})"
                                     f"{' OK' if pred == y[i] else ' X'}", fontsize=9)
    fig.suptitle("Attention Block maps (red = high attention)"); fig.tight_layout()
    fig.savefig(os.path.join(RUN, "attention_maps.png"), dpi=130); plt.show()


show_attention(model)

## 13. Predict a single image + clinical data

In [ ]:
def predict_one(model, image_path, age=None, sex="unknown", localization="unknown", dx_type="unknown"):
    row = pd.DataFrame([{"age": np.nan if age is None else age, "sex": sex,
                         "localization": localization, "dx_type": dx_type}])
    img = load_image(image_path).numpy()[None]
    probs = model.predict({"image": img, "clinical": transform_clinical(row, encoder)}, verbose=0)[0]
    for i in np.argsort(probs)[::-1]:
        print(f"  {CLASS_NAMES[i]:6s} {probs[i]:.4f}")


predict_one(model, lookup[test_df["image_id"].iloc[0]], age=test_df["age"].iloc[0],
            sex=test_df["sex"].iloc[0], localization=test_df["localization"].iloc[0])
print("true label:", test_df["dx"].iloc[0])

## 14. Ablation study (optional — each run takes a while)

In [ ]:
# run_experiment(use_attention=False)            # without attention block
# run_experiment(model_type="image_only")        # image branch only
# run_experiment(model_type="clinical_only")     # clinical MLP only
# run_experiment(backbone="mobilenetv2")         # lighter backbone

## 15. Compare all runs

In [ ]:
import glob
rows = []
for f in glob.glob(os.path.join(OUTPUT_DIR, "*", "test_metrics.json")):
    m = json.load(open(f)); m["run"] = os.path.basename(os.path.dirname(f)); rows.append(m)
pd.DataFrame(rows).set_index("run").round(4) if rows else "no runs yet"

## 16. Reload a trained model later (without retraining)

In [ ]:
def load_trained(run_path, which="best"):
    rc = json.load(open(os.path.join(run_path, "run_config.json")))
    m = build_model(rc["model_type"], rc["backbone"], rc["use_attention"], weights=None)
    p = weights_file(run_path, which)
    load_w(m, p if weights_exist(p) else weights_file(run_path, "final"))
    return m

# model = load_trained(os.path.join(OUTPUT_DIR, "densenet121_attn_fusion"))